# Stage 6: check a Parquet campaign

The Parquet equivalent of [4_check_database.ipynb](./4_check_database.ipynb).
Everything that was a DocumentDB collection is now an object on S3:

| 4_check_database | here |
|---|---|
| `stations` collection | `stations.parquet` |
| `picks` collection | `picks/network=/year=/month=/*.parquet` |
| `picks_record` collection | `complete/` and `progress/` |
| `sb_runs` collection | `runs/<run_id>.json` |
| — | `shards.jsonl`, the immutable work queue |
| — | `manifests/<job>.json`, what each job wrote |

Runs from anywhere — no VPC, no endpoint, no cluster to keep alive.

Three ways to read picks are shown, in increasing order of specificity:
§4 whole-dataset queries, §5 a single station, §6 a LIST-free path for services.

In [ ]:
import sys, json

sys.path.append("..")

import pandas as pd
import s3fs

from sb_catalog.src.s3_state import S3CampaignState

CAMPAIGN = "s3://<bucket>/<campaign>"

state = S3CampaignState(CAMPAIGN)
fs = s3fs.S3FileSystem()
root = CAMPAIGN[len("s3://"):]
assert "<bucket>" not in CAMPAIGN, "set CAMPAIGN"

## 1. Campaign progress

The equivalent of counting documents in `picks_record`, but it also tells you
how much work is left — which the collection could not.

- **complete** — shards finished, with their Parquet durable
- **in_flight** — claimed by a live worker, or abandoned and awaiting lease expiry
- **remaining** — not yet done

In [ ]:
p = state.progress()
print(p)
if p["total"]:
    print(f"\n{100 * p['complete'] / p['total']:.1f}% complete")

## 2. Stations

Was `sbc.get_stations()`.

In [ ]:
stations = state.get_stations()
print(f"{len(stations):,} stations, {stations.network_code.nunique()} networks")
stations.head()

## 3. What each job produced

The manifests are the audit trail: which station-days a job claimed, how many
picks each produced, and the exact objects written. This has no DocumentDB
equivalent — in 2025 you inferred it by counting rows.

In [ ]:
manifests = fs.glob(f"{root}/manifests/*.json")
print(f"{len(manifests)} job manifests")

rows = []
for m in manifests:
    d = json.loads(fs.cat(m))
    rows.append({"job": d["job_id"], "picks": d["n_picks"],
                 "station_days": d["station_days"],
                 "files": len(d.get("files", [])), "written": d["written_at"]})
jobs = pd.DataFrame(rows).sort_values("picks", ascending=False)
print(f"total picks across all jobs: {jobs.picks.sum():,}")
jobs.head()

## 4. Query the picks

The everyday path. `network`, `year` and `month` are **partition keys**, so
predicates on them skip whole prefixes rather than reading them; `tid` and
`conf` are pushed down to row-group statistics.

DuckDB is usually the least friction. pyarrow does the same thing if you would
rather stay in Arrow.

In [ ]:
import duckdb

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("SET s3_region='us-west-2';")   # region of the OUTPUT bucket

con.execute(f"""
    SELECT network, year, month, pha, count(*) AS picks,
           round(avg(conf), 3) AS mean_conf
    FROM read_parquet('{CAMPAIGN}/picks/**/*.parquet', hive_partitioning = true)
    GROUP BY 1, 2, 3, 4
    ORDER BY picks DESC
    LIMIT 20
""").df()

In [ ]:
# The same, in pyarrow.
import pyarrow.dataset as ds

dataset = ds.dataset(f"{root}/picks/", filesystem=fs,
                     format="parquet", partitioning="hive")
print(dataset.schema)

## 5. Picks for one station

The query 4_check_database did with `{"tid": ..., "peak": {...}}`.

In [ ]:
STATION = "CI.CLC."
YEAR, MONTH = 2019, 7

picks = con.execute(f"""
    SELECT tid, cha, pha, peak, conf, amp, amp_raw, rid
    FROM read_parquet('{CAMPAIGN}/picks/**/*.parquet', hive_partitioning = true)
    WHERE network = '{STATION.split('.')[0]}'
      AND year = {YEAR} AND month = {MONTH}
      AND tid = '{STATION}'
    ORDER BY peak
""").df()
print(f"{len(picks):,} picks")
print(picks.pha.value_counts().to_dict())
picks.head()

In [ ]:
# amp is Wood-Anderson displacement (metres); amp_raw is high-passed counts.
# NaN amp is expected and honest: a pick whose window fell inside a taper is
# not measurable, and reporting the suppressed value would be wrong rather
# than merely imprecise. See docs/amplitude_conventions.md.
print(f"amp set on {picks.amp.notna().sum():,} of {len(picks):,} picks")
picks[["conf", "amp", "amp_raw"]].describe()

## 6. Reading without a single LIST

§4 and §5 let the engine prune partitions, but pruning still **lists** the
prefixes it keeps. For a service, or any tight loop where request latency
matters, the manifests are an index: they record the exact object keys each job
wrote, which a reader cannot reconstruct from the file name because it carries
a content hash and a flush sequence number.

`shards.jsonl` -> `manifests/<shard_id>.json` -> object. GETs only.

In [ ]:
import pyarrow.parquet as pq

WANT, DAY = "CI.CLC.", "2019.187"

# 1 GET: the queue says which shard covers this station and day.
shards = [json.loads(l) for l in
          fs.cat(f"{root}/shards.jsonl").decode().splitlines() if l.strip()]
hits = [s for s in shards if WANT in s["stations"] and s["start"] <= DAY < s["end"]]
print(f"{len(hits)} shard(s) cover {WANT} on {DAY}")

for s in hits:
    # 1 GET: the manifest names the objects that shard produced.
    key = f"{root}/manifests/{s['shard_id']}.json"
    if not fs.exists(key):
        print(f"  {s['shard_id']}: not finished yet")
        continue
    manifest = json.loads(fs.cat(key))
    for f in manifest["files"]:
        if f["kind"] != "picks":
            continue
        # 1 GET: straight to the object. No LIST anywhere in this path.
        t = pq.read_table(f["path"], filesystem=fs,
                          columns=["tid", "pha", "peak", "conf", "amp"])
        print(f"  {f['path'].split('/')[-1]}: {t.num_rows:,} rows")

## 7. Provenance

Every pick carries `rid`, which resolves to the model, weight and thresholds
that produced it — what `sb_runs` did. This is how picks from different weights
stay separable inside one campaign.

In [ ]:
for r in fs.glob(f"{root}/runs/*.json")[:5]:
    print(json.loads(fs.cat(r)))

## 8. Re-running part of a campaign

There is no DANGER ZONE here, and that is the point. Parquet objects are
immutable and named after the shard that wrote them, so a re-run **overwrites
itself byte for byte** rather than duplicating rows — no `delete_many` before
re-picking, and no unique index to maintain.

To redo a shard, delete its completion marker and let a worker claim it again:

```python
fs.rm(f"{root}/complete/{shard_id}.json")
fs.rm(f"{root}/progress/{shard_id}.json")   # also discard partial progress
```

To discard a campaign entirely, delete the prefix. Nothing is running, so there
is nothing to stop first — unlike a DocumentDB cluster, which had to be deleted
or it kept billing.

In [ ]:
# # Redo one shard:
# shard_id = "2019001-2019021-abc123def456"
# for k in (f"{root}/complete/{shard_id}.json", f"{root}/progress/{shard_id}.json"):
#     if fs.exists(k):
#         fs.rm(k)
#         print("removed", k)